---
title: "Prepocesamiento"
format:
  html:
    code-fold: false
---

In [1]:
#| echo: false
import pandas as pd
import re
import requests
import json
import os
import glob

En el contexto del análisis de datos basados en registros generados por una aplicación web,
el preprocesamiento constituye una etapa fundamental dentro de la minería de datos. En este
caso, casi la totalidad de la información está en bruto dentro de la columna Descripción, por
lo que es necesario extraer de aquí todos los campos que nos serán útiles a posteriori para el
análisis y visualización de los mismos.

## Juntar los diferentes logs

In [2]:
csvs = glob.glob('..\logs/*.csv')

In [4]:
df = pd.concat([pd.read_csv(f,sep = ";", encoding="utf-8") for f in csvs], ignore_index=True)
pd.set_option('display.max_colwidth', None)


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008570 entries, 0 to 2008569
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   Fecha        object
 1   Usuario      object
 2   ID           int64 
 3   Acceso       object
 4   Descripción  object
 5   IP           object
dtypes: int64(1), object(5)
memory usage: 91.9+ MB


## Obtener los diferentes Eventos

El primer paso consiste en obtener la primera palabra de cada fila de dicha columna y conservar solo los valores únicos, lo que permite obtener una visión general de los distintos tipos
de eventos registrados.

In [4]:
lista = df["Descripción"].astype(str).str.split().str[0]

In [5]:
sorted(lista.unique())

['(Sesiones',
 'Aceptar',
 'Añadida',
 'Borrado',
 'Cambio',
 'Cargado',
 'Creado',
 'Cuenta',
 'Duración',
 'Error',
 'Generar',
 'Intento',
 'Limpiar',
 'Login',
 'Logout',
 'Nuevo',
 'Solicitud',
 'Visualizar',
 'nan']

In [6]:
len(lista.unique())

19

### Mapear los Eventos

Una vez identificados estos valores únicos, se realiza una observación
manual, filtrando las filas por cada uno de ellos, debido a que existen casos en los que distintos
eventos comparten una misma palabra inicial. Este proceso de exploración también ayuda a
detectar patrones en la columna Descripción.

In [7]:
mapeo_eventos = {
    '(Sesiones': lambda x: "CONEXION" if re.search(r'\bconectada\b', x.lower()) else "DESCONEXION",
    'Aceptar': "ACEPTAR_PRIVACIDAD",       
    'Añadida': "AÑADIR_PIEZA",
    'Borrado': "BORRAR_DISEÑO",          
    'Cambio': "CAMBIO_CONTRASEÑA",              
    'Cargado': "CARGAR_DISENO",
    'Creado': "CREAR_DISENO",
    'Cuenta': "CUENTA",
    'Duración': "DURACION",             
    'Error': "ERROR_SISTEMA",          
    'Generar': lambda x: "GEN_PRESUPUESTO" if re.search(r'\bpresupuesto\b', x.lower()) else "GEN_PEDIDO",
    'Intento': "INTENTO_RESET",       
    'Limpiar': "LIMPIAR_DATOS",        
    'Login': "LOGIN",
    'Logout': "LOGOUT",
    'Nuevo': "NUEVO_LOGO",              
    'Solicitud': "SOLICITUD_RESET",            
    'Visualizar': lambda x: "VER_PRESUPUESTO" if re.search(r'\bpresupuesto\b', x.lower()) else "VER_PEDIDO",
}

In [8]:
def categorizar_evento(descripcion):
    descripcion = str(descripcion)
    primera_palabra = descripcion.split()[0]
    
    if primera_palabra in mapeo_eventos:
        if callable(mapeo_eventos[primera_palabra]):
            return mapeo_eventos[primera_palabra](descripcion)
        else:
            return mapeo_eventos[primera_palabra]
    else:
        return "OTRO"

In [ ]:
df["Evento"] = df["Descripción"].apply(categorizar_evento)

In [10]:
df[df['Evento'] == 'AÑADIR_PIEZA'][['Descripción', 'Evento']].head(10)

,Descripción,Evento
2,Añadida la pieza SIN-021 con id 2068 del catál...,AÑADIR_PIEZA
3,Añadida la pieza SOF-022 con id 4684 del catál...,AÑADIR_PIEZA
4,Añadida la pieza BA521A con id 319127 del catá...,AÑADIR_PIEZA
6,Añadida la pieza 1252 con id 81570 del catálog...,AÑADIR_PIEZA
7,Añadida la pieza CNI406T con id 359241 del cat...,AÑADIR_PIEZA
9,Añadida la pieza BA178 con id 318748 del catál...,AÑADIR_PIEZA
10,Añadida la pieza BA510A con id 319121 del catá...,AÑADIR_PIEZA
12,Añadida la pieza ED-LIB-12 con id 20055 del ca...,AÑADIR_PIEZA
20,Añadida la pieza 5700 con id 310452 del catálo...,AÑADIR_PIEZA
34,Añadida la pieza ED-TVT-01 con id 2064 del cat...,AÑADIR_PIEZA


### Obtener columnas a partir de la descripción

Posteriormente, estos patrones se utilizan para definir expresiones regulares que permiten generar nuevas columnas.

In [11]:
def extraer_columnas(row):
    evento = row['Evento']
    desc = row['Descripción']
    
    if pd.isna(desc):
        return {}

    if evento == 'AÑADIR_PIEZA':
        catalogo_completo = buscar(r'catálogo (.*?) en el diseño', desc)
        if catalogo_completo and ' - ' in catalogo_completo:
            fabricante, catalogo = catalogo_completo.split(' - ', 1)
        else:
            fabricante, catalogo = None, catalogo_completo
        
        return {
            'pieza': buscar(r'pieza ([\w-]+)', desc),
            'piezaid': buscar(r'id (\d+)', desc),
            'fabricante': fabricante,
            'catalogo': catalogo,
            'diseñoid': buscar(r'diseño con id: (-?\d+)', desc)
        }

    elif evento in ['LOGIN', 'LOGOUT', 'SOLICITUD_RESET', 'ACEPTAR_PRIVACIDAD', 'CAMBIO_CONTRASEÑA']:
        return {'userid': buscar(r'userID: (\w+)', desc)}

    elif evento == 'GEN_PEDIDO':
        return {
            'diseñoid': buscar(r'diseño con id: (-?\d+)', desc),
            'fabricante': buscar(r'fabricante (.*)', desc)
        }

    elif evento in ['CREAR_DISENO', 'CARGAR_DISENO', 'GEN_PRESUPUESTO', 'VER_PRESUPUESTO', 'ERROR_SISTEMA', 'BORRAR_DISEÑO', 'VER_PEDIDO']:
        return {'diseñoid': buscar(r'diseño con id: (-?\d+)', desc)}

    elif evento == 'INTENTO_RESET':
        return {'email': buscar(r'email[:\s]+(\S+)', desc)}

    elif evento == 'DURACION':
        return {'minutos': buscar(r'Duración de la sesión:\s*([\d.]+)', desc)}

    elif evento == 'NUEVO_LOGO':
        return {'logo': buscar(r'logo[:\s]+(\w+)', desc)}
    
    elif evento in ['CONEXION', 'DESCONEXION']:
        return {'nsesiones': buscar(r'activas (\d+)', desc)}

    return {}

def buscar(patron, texto):
    match = re.search(patron, texto)
    return match.group(1) if match else None

In [12]:
df_extra = df.apply(extraer_columnas, axis=1, result_type='expand')

In [13]:
df = pd.concat([df, df_extra], axis=1)

## Obtener columnas geograficas a partir de una API

El conjunto de datos inicial nos provee las IPs de los diferentes usuarios que acceden a
la aplicación. Para enriquecer esta información, se ha optado por utilizar una API externa,
ipinfo.io, que permite obtener la ubicación geográfica aproximada asociada a cada IP, que nos
proporciona la ciudad, la región, el país y las coordenadas (latitud y longitud).

In [14]:
def geolocalizar(ip):
    url= f'https://ipinfo.io/{ip}?token=aff46df76e5e05'
    try:
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()
        lat_str, lng_str = data['loc'].split(',')
        lat, lng = float(lat_str), float(lng_str)
        return {
            'city': data.get('city'),
            'region': data.get('region'),
            'country': data.get('country'),
            'lat': lat,
            'lng': lng
        }
    except Exception as e:
        print(f'Error con la ip: {ip}')
        return {'city': None, 'region': None, 'country': None, 'lat': None, 'lng': None}

In [15]:
ips = df['IP'].unique()

La información
recuperada tras realizar la consulta a la API se almacena en un archivo JSON, que actúa como
caché para evitar realizar peticiones repetidas.

In [16]:
geo = '..\data\geolocalizacion.json'

if os.path.exists(geo) and os.path.getsize(geo) > 0:
    with open(geo, 'r') as f:
        geolocalizacion = json.load(f)
else:
    geolocalizacion = {}

for ip in ips:
    if ip not in geolocalizacion:
        geolocalizacion[ip] = geolocalizar(ip) 

with open('..\data\geolocalizacion.json', 'w') as f:
    json.dump(geolocalizacion, f)

Posteriormente, estos datos se incorporan a la
tabla mediante un mapeo que accede al archivo JSON generado, lo que permite la creación de
nuevas columnas con la información geográfica correspondiente.


In [17]:
df['city'] = df['IP'].map(lambda ip: geolocalizacion.get(ip, {}).get('city'))
df['region'] = df['IP'].map(lambda ip: geolocalizacion.get(ip, {}).get('region'))
df['country'] = df['IP'].map(lambda ip: geolocalizacion.get(ip, {}).get('country'))
df['lat'] = df['IP'].map(lambda ip: geolocalizacion.get(ip, {}).get('lat'))
df['lng'] = df['IP'].map(lambda ip: geolocalizacion.get(ip, {}).get('lng'))

## Cambiar los tipos de las columnas

Se procede a ajustar los tipos de datos con el fin de optimizar el consumo de memoria,
considerando las limitaciones de la máquina. Además, se ha decidido eliminar
de la columna Usuario el identificador, puesto que no aporta información relevante y se prefiere mostrar el nombre limpio. También se ha especificado que Fecha es un datetime con el
formato %Y/ %m/ %d %H: %M: %S y se han creado dos columnas adicionales (Hora y NombreDia), para facilitar la realización de algunas visualizaciones.

In [ ]:
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
df['Hora'] = df['Fecha'].dt.hour
df['NombreDia'] = df['Fecha'].dt.day_name()
df = df.sort_values('Fecha').reset_index(drop=True)
df['Usuario'] = df['Usuario'].str.replace(r'^#\d+\s*-\s*', '', regex=True)

In [ ]:
df = df.astype({
    'Evento': 'string', 'nsesiones': 'Int64',
    'pieza': 'string', 'fabricante': 'string',
    'catalogo': 'string', 'minutos': 'float32',
    'logo': 'string', 'email': 'string',
    'city': 'category', 'region': 'category',
    'country': 'category', 'lat': 'float32',
    'lng': 'float32'
})

In [ ]:
df.head(10)

,Fecha,Usuario,ID,Acceso,Descripción,IP,Evento,diseñoid,nsesiones,pieza,...,minutos,logo,email,city,region,country,lat,lng,Hora,NombreDia
0,2023-11-24 12:08:35,Rafael Marcos Luque Baena,2883799,OK,Limpiar LOG,150.214.58.151,LIMPIAR_DATOS,NaN,<NA>,<NA>,...,NaN,<NA>,<NA>,Málaga,Andalusia,ES,36.720200,-4.4203,12,Friday
1,2023-11-24 12:08:38,BIJOTA ALZARIAK. S.L.,2883802,OK,Añadida la pieza ED-ALF-02 con id 20020 del ca...,88.14.10.127,AÑADIR_PIEZA,55976,<NA>,ED-ALF-02,...,NaN,<NA>,<NA>,Valencia,Valencia,ES,39.473900,-0.3797,12,Friday
2,2023-11-24 12:08:38,BIJOTA ALZARIAK. S.L.,2883800,OK,Añadida la pieza ED-LIB-14 con id 20057 del ca...,88.14.10.127,AÑADIR_PIEZA,55976,<NA>,ED-LIB-14,...,NaN,<NA>,<NA>,Valencia,Valencia,ES,39.473900,-0.3797,12,Friday
3,2023-11-24 12:08:38,MUEBLES MEDINA 2015 S.L.,2883801,OK,Añadida la pieza 607354G con id 219337 del cat...,185.210.19.174,AÑADIR_PIEZA,56317,<NA>,607354G,...,NaN,<NA>,<NA>,Torrevieja,Valencia,ES,37.978699,-0.6822,12,Friday
4,2023-11-24 12:08:39,BIJOTA ALZARIAK. S.L.,2883803,OK,Añadida la pieza EE-PBA-03 con id 19976 del ca...,88.14.10.127,AÑADIR_PIEZA,55976,<NA>,EE-PBA-03,...,NaN,<NA>,<NA>,Valencia,Valencia,ES,39.473900,-0.3797,12,Friday
5,2023-11-24 12:08:42,MOBLES SALVANY,2883804,OK,Añadida la pieza DC-10 con id 396700 del catál...,80.58.139.181,AÑADIR_PIEZA,56555,<NA>,DC-10,...,NaN,<NA>,<NA>,Premià de Mar,Catalonia,ES,41.492100,2.3652,12,Friday
6,2023-11-24 12:08:43,MUEBLES NIETO,2883805,OK,Añadida la pieza V1041606D con id 49734 del ca...,81.38.17.247,AÑADIR_PIEZA,55123,<NA>,V1041606D,...,NaN,<NA>,<NA>,Ávila,Castille and León,ES,40.657200,-4.6995,12,Friday
7,2023-11-24 12:08:43,MUBAK MARESME,2883806,OK,Cargado el diseño con id: 56398,149.6.200.198,CARGAR_DISENO,56398,<NA>,<NA>,...,NaN,<NA>,<NA>,Barcelona,Catalonia,ES,41.388802,2.1590,12,Friday
8,2023-11-24 12:08:44,RUIZ MORAGA FRANCISCO,2883807,OK,Cargado el diseño con id: 56313,37.132.236.134,CARGAR_DISENO,56313,<NA>,<NA>,...,NaN,<NA>,<NA>,Bolaños de Calatrava,Castille-La Mancha,ES,38.906898,-3.6635,12,Friday
9,2023-11-24 12:08:46,MUBAK MARESME,2883808,OK,Cargado el diseño con id: 56398,149.6.200.198,CARGAR_DISENO,56398,<NA>,<NA>,...,NaN,<NA>,<NA>,Barcelona,Catalonia,ES,41.388802,2.1590,12,Friday


In [21]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2008570 entries, 0 to 2008569
Data columns (total 24 columns):
 #   Column       Dtype         
---  ------       -----         
 0   Fecha        datetime64[ns]
 1   Usuario      object        
 2   ID           int64         
 3   Acceso       object        
 4   Descripción  object        
 5   IP           object        
 6   Evento       string        
 7   diseñoid     object        
 8   nsesiones    Int64         
 9   pieza        string        
 10  piezaid      object        
 11  fabricante   string        
 12  catalogo     string        
 13  userid       object        
 14  minutos      float32       
 15  logo         string        
 16  email        string        
 17  city         category      
 18  region       category      
 19  country      category      
 20  lat          float32       
 21  lng          float32       
 22  Hora         int32         
 23  NombreDia    object        
dtypes: Int64(1), category(3)

## Guardar dataframes

In [22]:
df.to_parquet("..\data\logs.parquet",index=False)

In [23]:
df.to_csv("..\data\logs.csv",index=False)